### TF-IDF 분석 (국내 KR / 글로벌 GB)

## TF-IDF analysis 개요

감성분류가 완료된 리뷰(`kr_sentiment.csv`, `gb_sentiment.csv`)를 대상으로, **긍정/부정 리뷰 간 TF-IDF 값 차이(gap)** 를 계산해 각 그룹에서 상대적으로 더 자주/중요하게 언급되는 키워드를 추출합니다.

### 처리 순서
1. 확신도(`sentiment_score`) 0.8 이상 리뷰만 필터링
2. 형태소 분석(KR: KoNLPy Okt / GB: NLTK POS tagger)으로 명사(+영어는 형용사 포함) 추출
3. 사전 정의된 **정규화 사전**(비슷한 표현을 하나의 키워드로 통합)과 **불용어 사전**을 적용
4. TF-IDF 벡터화 후 긍정/부정 리뷰 그룹 간 평균 TF-IDF 차이(gap) 계산
5. 워드클라우드 + 막대그래프로 시각화

### ⚠️ 참고 및 실행 전 준비사항
- 정규화 사전(`norm_groups`, `en_norm_groups`)은 원본 리뷰를 반복 탐색하며 만든 결과물입니다. 이 사전을 만드는 과정(특정 단어·리뷰 검색, 빈도 반복 확인)은 본 파일에서는 생략하고, **최종적으로 사용한 사전과 실제 분석 로직만 정리**했습니다.
- 본 파일은 `stopwords.txt`, `stopwords_gb.txt`, `stopwords_gb_sentiment.txt` 파일을 참조합니다. 해당 불용어 사전은 분석 대상에 따라 달라지는 값이므로 본 저장소에는 포함하지 않습니다.
- 한국어 형태소 분석(KoNLPy Okt)은 Java(JDK) 설치가 필요합니다.
- 영어 처리(NLTK)는 최초 1회 아래 리소스를 다운로드해야 합니다.
  ```python
  import nltk
  nltk.download('punkt')
  nltk.download('averaged_perceptron_tagger')
  nltk.download('wordnet')
  ```
- 워드클라우드에 쓰인 한글 폰트 경로(`malgun.ttf`)는 Windows 기준입니다. Mac/Linux에서는 본인 환경에 맞는 폰트 경로로 수정해야 합니다.


## 1. 국내(KR) 리뷰 TF-IDF 분석

In [ ]:
#step 1. 모듈
from konlpy.tag import Okt
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

okt = Okt()


In [ ]:
#step 2. 파일 불러오기 (경로는 실행 환경에 맞게 수정)
kr = pd.read_csv('kr_sentiment.csv')
print(f'전체 데이터 수 : {len(kr)}건')
print(f'컬럼명 : {kr.columns.tolist()}')


In [ ]:
#step 3. 확신도 0.8 기준으로 필터링 + 긍정/부정 분리
kr_filtered = kr[kr['sentiment_score'] >= 0.8].copy()
print(f'sentiment score 0.8 이상 리뷰 수 : {len(kr_filtered)}건')

pos_all = kr_filtered[kr_filtered['sentiment_label'] == '긍정']
neg_all = kr_filtered[kr_filtered['sentiment_label'] == '부정']
print(f'긍정 리뷰수 : {len(pos_all)}건 / 부정 리뷰수 : {len(neg_all)}건')


In [ ]:
# 상품 코드 → 상품명 매핑 (예시. 실제 프로젝트에서는 10개 상품의 goods_no를 매핑하여 사용)
product_map = {
    'A000000000001': '1_product_a',
    'A000000000002': '2_product_b',
    # ... 나머지 상품 매핑 생략
}

kr_filtered['product_name'] = kr_filtered['goods_no'].map(product_map)

In [ ]:
# 정규화 사전 (반복 탐색 끝에 확정된 최종 버전) + 불용어 사전
# 비슷한 표현(오타, 활용형 등)을 하나의 대표 키워드로 통합해 TF-IDF 계산 시 흩어지지 않도록 함
norm_groups = {
    '발림성' : {'발라', '발랏', '발랏는', '발랏는데', '발랏다', '발랏을때', '발랏을땬', '발랴', '발리',
                '발리나', '발린', '발림', '발링', '성도'},
    '눈시림' : {'시려웠', '시렵거', '시렵던', '시리', '시림'},
    '밀림'   : {'밀리', '밀어내기'},
    '밀착력' : {'밀착', '밀착렫', '밀철'},
    '덧'     : {'덧바름'},
    '끈적임' : {'끈', '끈끈', '끈끈함', '끈덕', '끈덕끈', '끈쩍', '끈쩍끈쩍', '끈쩍임', '적임'},
    '화끈거림' : {'화끈', '화끈거렸', '후끈'},
    '선크림' : {'썬크림'},
    '백탁'   : {'탁', '백탁현상'},
    '가성비' : {'가성'},
    '사용감' : {'용감'},
    '무기자차' : {'무기'},
    '유기자차' : {'유기'},
    '수부지' : {'부지'},
    '사은품' : {'사은'},
    '복합성' : {'복합'},
    '유지력' : {'유지', '유지됨'},
    '지속력' : {'지속'},
    '트러블' : {'뽀로지', '뾰루지', '뽀루지'},
    '지성'   : {'지성만', '지성은', '지성이면', '지성인'},
    '쿨링감' : {'쿨롱', '쿨링', '쿨링금'}
}

# 정규화 사전 → lookup용 딕셔너리로 변환
norm_dict = {}
for target, sources in norm_groups.items():
    for source in sources:
        norm_dict[source] = target
print(f'정규화 사전 총 {len(norm_dict)}개 항목')

# 불용어 사전 (별도 파일 필요 - 경로는 실행 환경에 맞게 수정)
with open('stopwords.txt', 'r', encoding='utf-8') as f:
    stopwords = set(f.read().splitlines())
stopwords = {w for w in stopwords if w.strip()}
print(f'불용어 {len(stopwords)}개')


In [ ]:
# 형태소 분석기 (명사만 추출 + 정규화 + 불용어 제거)
def okt_tokenizer(text):
    morphs = okt.pos(str(text), stem=True, norm=True)
    tokens = []
    for word, pos in morphs:
        if pos in ['Noun']:
            word = norm_dict.get(word, word)
            if word not in stopwords:
                tokens.append(word)
    return tokens


In [ ]:
# TF-IDF 계산 함수
def build_tfidf(docs):
    vectorizer = TfidfVectorizer(
        tokenizer=okt_tokenizer,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 1),
        max_df=0.9,
        min_df=1
    )
    tfidf_matrix = vectorizer.fit_transform(docs)
    feature_names = vectorizer.get_feature_names_out()
    return tfidf_matrix, feature_names

# 단어별 평균 TF-IDF 계산
def aggregate_tfidf(tfidf_matrix, feature_names):
    arr = tfidf_matrix.toarray()
    avg_scores = np.mean(arr, axis=0)
    return pd.Series(avg_scores, index=feature_names)

# 제품별 긍정/부정 TF-IDF gap 분석
def analyze_product(product_name, top_n=20):
    print(f'\n{"-"*10} 제품: {product_name} {"-"*10}')

    df = kr_filtered[kr_filtered['product_name'] == product_name]
    pos_docs = df[df['sentiment_label'] == '긍정']['review_clean'].tolist()
    neg_docs = df[df['sentiment_label'] == '부정']['review_clean'].tolist()

    print(f'긍정: {len(pos_docs)}건 / 부정: {len(neg_docs)}건')

    tfidf_p, feature_p = build_tfidf(pos_docs)
    tfidf_n, feature_n = build_tfidf(neg_docs)

    agg_p = aggregate_tfidf(tfidf_p, feature_p)
    agg_n = aggregate_tfidf(tfidf_n, feature_n)

    common_words = set(agg_p.index) & set(agg_n.index)
    df_compare = pd.DataFrame({
        'tfidf_pos': agg_p.loc[list(common_words)],
        'tfidf_neg': agg_n.loc[list(common_words)]
    }).fillna(0)
    df_compare['gap'] = df_compare['tfidf_pos'] - df_compare['tfidf_neg']

    top_pos = df_compare.sort_values('gap', ascending=False).head(top_n)
    top_neg = df_compare.sort_values('gap', ascending=True).head(top_n)

    print(f'\n[공통 단어 gap 분석 — 장점 Top {top_n}]')
    for word, row in top_pos.iterrows():
        print(f'  {word}: {row["gap"]:.4f}')

    print(f'\n[공통 단어 gap 분석 — 단점 Top {top_n}]')
    for word, row in top_neg.iterrows():
        print(f'  {word}: {row["gap"]:.4f}')

    return df_compare


In [ ]:
# 제품별 TF-IDF 실행 (10개 상품 반복)
product_list = ['1_product_a', '2_product_b',,,
                # 상품명 생략
                ]

all_results = {}

for product in product_list:
    result = analyze_product(product, top_n=20)
    all_results[product] = result
    print()

print('='*30)
print('제품별 TF-IDF 국내 리뷰 분석 완료')


In [ ]:
# 전체 리뷰 기준 TF-IDF gap 분석
def analyze_total(top_n=30):
    print('\n' + '='*40)
    print('전체 국내 리뷰 TF-IDF 분석')
    print('='*40)

    pos_docs = kr_filtered[kr_filtered['sentiment_label'] == '긍정']['review_clean'].tolist()
    neg_docs = kr_filtered[kr_filtered['sentiment_label'] == '부정']['review_clean'].tolist()

    print(f'긍정: {len(pos_docs):,}건 / 부정: {len(neg_docs):,}건')

    tfidf_p, feature_p = build_tfidf(pos_docs)
    tfidf_n, feature_n = build_tfidf(neg_docs)

    agg_p = aggregate_tfidf(tfidf_p, feature_p)
    agg_n = aggregate_tfidf(tfidf_n, feature_n)

    common_words = set(agg_p.index) & set(agg_n.index)
    df_compare = pd.DataFrame({
        'tfidf_pos': agg_p.loc[list(common_words)],
        'tfidf_neg': agg_n.loc[list(common_words)]
    }).fillna(0)
    df_compare['gap'] = df_compare['tfidf_pos'] - df_compare['tfidf_neg']

    print(f'\n[전체 — 장점 Top {top_n}]')
    for word, row in df_compare.sort_values('gap', ascending=False).head(top_n).iterrows():
        print(f'  {word}: {row["gap"]:.4f}')

    print(f'\n[전체 — 단점 Top {top_n}]')
    for word, row in df_compare.sort_values('gap', ascending=True).head(top_n).iterrows():
        print(f'  {word}: {row["gap"]:.4f}')

    return df_compare

# 실행
df_total = analyze_total(top_n=30)


### 워드클라우드 · 막대그래프 시각화 (KR)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.font_manager as fm
from wordcloud import WordCloud

# 한글 폰트 경로 (Windows 기준 — Mac/Linux는 본인 환경에 맞는 폰트 경로로 수정 필요)
font_location = r'C:\Windows\Fonts\malgun.ttf'
font_name = fm.FontProperties(fname=font_location).get_name()
matplotlib.rc('font', family=font_name)   # 제목, 축 등 한글 폰트 지정

# 워드클라우드 이미지 내부 폰트
font_path = font_location


In [ ]:
#색상 함수
def make_color_func(colormap_name, vmin=0.45, vmax=0.88):
    cmap = cm.get_cmap(colormap_name)
    def color_func(word, font_size, position, orientation, random_state=None, **kwargs):
        val = random_state.uniform(vmin, vmax) if random_state else 0.7
        r, g, b, _ = cmap(val)
        return f'rgb({int(r*255)}, {int(g*255)}, {int(b*255)})'
    return color_func

#워드클라우드 함수
def make_wordcloud(freq_dict, colormap='Blues'):
    if not freq_dict:
        return None
    wc = WordCloud(
        font_path=font_path,
        background_color='white',
        color_func=make_color_func(colormap),
        width=800,
        height=500,
        prefer_horizontal=0.9,
        relative_scaling=0.65,
        collocations=False,
        margin=4,
        random_state=10
    ).generate_from_frequencies(freq_dict)
    return wc

def plot_wordcloud(df_compare, title, top_n=30):
    """
    df_compare : analyze_product() 또는 analyze_total() 반환값
    """
    pos_words = df_compare.sort_values('gap', ascending=False).head(top_n)
    neg_words = df_compare.sort_values('gap', ascending=True).head(top_n)

    pos_dict = {word: abs(row['gap']) for word, row in pos_words.iterrows()}
    neg_dict = {word: abs(row['gap']) for word, row in neg_words.iterrows()}

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)

    wc_pos = make_wordcloud(pos_dict, 'Blues')
    wc_neg = make_wordcloud(neg_dict, 'Reds')

    if wc_pos:
        axes[0].imshow(wc_pos, interpolation='bilinear')
    axes[0].set_title('긍정 리뷰 키워드', fontsize=13, pad=10)
    axes[0].axis('off')

    if wc_neg:
        axes[1].imshow(wc_neg, interpolation='bilinear')
    axes[1].set_title('부정 리뷰 키워드', fontsize=13, pad=10)
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
# 1) 전체 리뷰 워드클라우드
plot_wordcloud(df_total, '전체 국내 선크림 리뷰 — 장단점 키워드')


In [ ]:
#막대그래프 시각화
def plot_tfidf_bar(df_compare, title, top_n=15):
    top_pos = df_compare.sort_values('gap', ascending=False).head(top_n)
    top_neg = df_compare.sort_values('gap', ascending=True).head(top_n)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(title, fontsize=15, fontweight='bold')

    axes[0].barh(top_pos.index[::-1], top_pos['gap'][::-1], color='steelblue')
    axes[0].set_title('긍정 키워드 Top 15', fontsize=13)
    axes[0].set_xlabel('TF-IDF gap')

    axes[1].barh(top_neg.index, top_neg['gap'].abs(), color='tomato')
    axes[1].set_title('부정 키워드 Top 15', fontsize=13)
    axes[1].set_xlabel('TF-IDF gap')

    plt.tight_layout()
    plt.show()

# 실행
plot_tfidf_bar(df_total, '전체 국내 선크림 리뷰 — 장단점 키워드')


In [ ]:
# 2) 제품별 워드클라우드 (예시 3개 상품)
target_products = ['1_roundlab', '4_boj_tinted', '5_mediheal']

for product in target_products:
    df_result = analyze_product(product, top_n=30)
    plot_wordcloud(df_result, f'{product} — 장단점 키워드')


## 2. 글로벌(GB) 리뷰 TF-IDF 분석

In [ ]:
# step 1. 모듈
import nltk
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

lemmatizer = WordNetLemmatizer()

# 파일 불러오기 (경로는 실행 환경에 맞게 수정)
gb = pd.read_csv('gb_sentiment.csv')

# 확신도 0.8 이상 필터링
gb_filtered = gb[gb['sentiment_score'] >= 0.8].copy()
print(f'sentiment_score 0.8 이상: {len(gb_filtered)}건')

pos_all = gb_filtered[gb_filtered['sentiment_label'] == '긍정']
neg_all = gb_filtered[gb_filtered['sentiment_label'] == '부정']
print(f'긍정: {len(pos_all)}건 / 부정: {len(neg_all)}건')


In [ ]:
# 불용어 사전 (별도 파일 필요 - 경로는 실행 환경에 맞게 수정)
with open('stopwords_gb.txt', 'r', encoding='utf-8') as f:
    base_stopwords = set(f.read().splitlines())
    base_stopwords = {w.strip() for w in base_stopwords if w.strip()}

with open('stopwords_gb_sentiment.txt', 'r', encoding='utf-8') as f:
    sentiment_stopwords = set(f.read().splitlines())
    sentiment_stopwords = {w.strip() for w in sentiment_stopwords if w.strip()}

# TF-IDF용 불용어 = 공통 불용어 + 감성 관련 불용어
tfidf_stopwords = base_stopwords | sentiment_stopwords

# 영어 정규화 사전 (반복 탐색 끝에 확정된 최종 버전)
en_norm_groups = {
    'white_cast'   : {'white', 'cast', 'whitecast', 'casting', 'overcast'},
    'lightweight'  : {'light', 'lightweighted', 'litghweight', 'weightless'},
    'sticky'       : {'stickiness', 'tacky', 'stick', 'sticker', 'stickey', 'stickness'},
    'greasy'       : {'greasiness', 'grease'},
    'moisturizing' : {'moisturizes', 'moisturizer', 'moisturising', 'moisturiser', 'moisture', 'moist'},
    'hydrating'    : {'hydration', 'hydrated'},
    'scent'        : {'smell', 'fragrance'},
    'acne'         : {'breakout', 'pimple', 'blemish', 'prone'},
    'irritation'   : {'irritate', 'irritated', 'irritant', 'irritates', 'irritating'},
    'packaging'    : {'package', 'pack'},
    'smooth'       : {'smoothly', 'spread', 'blend'},
    'oily'         : {'oil', 'oiler', 'oiliness', 'oily'},
    'transparency' : {'transparent'},
    'brightening'  : {'bright', 'brightens', 'brighter', 'brightness'},
    'breakout'     : {'break', 'breakouts', 'outbreak'},
    'absorption'   : {'absorb', 'absorbance', 'absorbed', 'absorbency', 'absorbent',
                       'absorber', 'absorbes', 'absorbing', 'absorbs'}
}

en_norm_dict = {}
for target, sources in en_norm_groups.items():
    for source in sources:
        en_norm_dict[source] = target


In [ ]:
# 상품 코드 → 상품명 매핑 (예시. 실제 프로젝트에서는 10개 상품의 prdtNo를 매핑하여 사용)
product_map_gb = {
        'GA000000001': '1_product_a',
        'GA000000002': '2_product_b',
        # ... 나머지 상품 매핑 생략
}

gb_filtered['product_name'] = gb_filtered['goods_no'].map(product_map_gb)

# TF-IDF 토크나이저 (명사/형용사 추출 + lemmatize + 정규화 + 불용어 제거)
def en_tokenizer_tfidf(text):
    tokens = word_tokenize(str(text).lower())
    tagged = pos_tag(tokens)
    result = []
    for word, tag in tagged:
        if tag.startswith('NN') or tag.startswith('JJ'):
            if tag.startswith('NN'):
                lemma = lemmatizer.lemmatize(word, pos='n')
            else:
                lemma = lemmatizer.lemmatize(word, pos='a')
            lemma = en_norm_dict.get(lemma, lemma)
            if len(lemma) >= 2 and lemma not in tfidf_stopwords:
                result.append(lemma)
    return result

# TF-IDF 계산 함수
def build_tfidf_en(docs):
    vectorizer = TfidfVectorizer(
        tokenizer=en_tokenizer_tfidf,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 1),
        max_df=0.9,
        min_df=2
    )
    tfidf_matrix = vectorizer.fit_transform(docs)
    feature_names = vectorizer.get_feature_names_out()
    return tfidf_matrix, feature_names

def aggregate_tfidf_en(tfidf_matrix, feature_names):
    arr = tfidf_matrix.toarray()
    avg_scores = np.mean(arr, axis=0)
    return pd.Series(avg_scores, index=feature_names)


In [ ]:
# 전체 리뷰 기준 TF-IDF gap 분석
def analyze_total_en(top_n=30):
    print('\n' + '='*40)
    print('전체 글로벌 리뷰 TF-IDF 분석')
    print('='*40)

    pos_docs = gb_filtered[gb_filtered['sentiment_label'] == '긍정']['review_clean'].tolist()
    neg_docs = gb_filtered[gb_filtered['sentiment_label'] == '부정']['review_clean'].tolist()
    print(f'긍정: {len(pos_docs):,}건 / 부정: {len(neg_docs):,}건')

    tfidf_p, feature_p = build_tfidf_en(pos_docs)
    tfidf_n, feature_n = build_tfidf_en(neg_docs)

    agg_p = aggregate_tfidf_en(tfidf_p, feature_p)
    agg_n = aggregate_tfidf_en(tfidf_n, feature_n)

    common_words = set(agg_p.index) & set(agg_n.index)
    df_compare = pd.DataFrame({
        'tfidf_pos': agg_p.loc[list(common_words)],
        'tfidf_neg': agg_n.loc[list(common_words)]
    }).fillna(0)
    df_compare['gap'] = df_compare['tfidf_pos'] - df_compare['tfidf_neg']

    print(f'\n[전체 긍정 Top {top_n}]')
    for word, row in df_compare.sort_values('gap', ascending=False).head(top_n).iterrows():
        print(f'  {word}: {row["gap"]:.4f}')

    print(f'\n[전체 부정 Top {top_n}]')
    for word, row in df_compare.sort_values('gap', ascending=True).head(top_n).iterrows():
        print(f'  {word}: {row["gap"]:.4f}')

    return df_compare

# 제품별 긍정/부정 TF-IDF 분석
def analyze_product_en(product_name, top_n=20):
    print(f'\n{"="*40}')
    print(f'제품: {product_name}')
    print(f'{"="*40}')

    df = gb_filtered[gb_filtered['product_name'] == product_name]
    pos_docs = df[df['sentiment_label'] == '긍정']['review_clean'].tolist()
    neg_docs = df[df['sentiment_label'] == '부정']['review_clean'].tolist()
    print(f'긍정: {len(pos_docs):,}건 / 부정: {len(neg_docs):,}건')

    if len(pos_docs) < 5 or len(neg_docs) < 5:
        print('⚠️ 리뷰 수 부족 → 스킵')
        return None

    results = {}
    for sentiment, docs in [('긍정', pos_docs), ('부정', neg_docs)]:
        tfidf_matrix, feature_names = build_tfidf_en(docs)
        agg = aggregate_tfidf_en(tfidf_matrix, feature_names)
        top = agg.sort_values(ascending=False).head(top_n)
        results[sentiment] = top

        print(f'\n[{sentiment} Top {top_n}]')
        for word, score in top.items():
            print(f'  {word}: {score:.4f}')

    return results


In [ ]:
# 실행: 전체 + 제품별
target_products_gb = [
            '1_product_a', '2_product_b',,, 
            #상품명 생략
                      ]
all_results_gb = {}

print('전체 글로벌 리뷰 TF-IDF')
pos_docs_all = gb_filtered[gb_filtered['sentiment_label'] == '긍정']['review_clean'].tolist()
neg_docs_all = gb_filtered[gb_filtered['sentiment_label'] == '부정']['review_clean'].tolist()

tfidf_p, feature_p = build_tfidf_en(pos_docs_all)
tfidf_n, feature_n = build_tfidf_en(neg_docs_all)
agg_p = aggregate_tfidf_en(tfidf_p, feature_p)
agg_n = aggregate_tfidf_en(tfidf_n, feature_n)

all_results_gb['total'] = {
    '긍정': agg_p.sort_values(ascending=False).head(20),
    '부정': agg_n.sort_values(ascending=False).head(20)
}

print('[전체 긍정 Top 20]')
for word, score in all_results_gb['total']['긍정'].items():
    print(f'  {word}: {score:.4f}')

print('\n[전체 부정 Top 20]')
for word, score in all_results_gb['total']['부정'].items():
    print(f'  {word}: {score:.4f}')

for product in target_products_gb:
    result = analyze_product_en(product, top_n=20)
    if result is not None:
        all_results_gb[product] = result


### 워드클라우드 시각화 (GB)

In [ ]:
#색상 함수 및 워드클라우드 함수 (영문 - 별도 폰트 지정 불필요)
def make_wordcloud_en(freq_dict, colormap='Blues'):
    if not freq_dict:
        return None
    wc = WordCloud(
        background_color='white',
        color_func=make_color_func(colormap),
        width=800,
        height=500,
        prefer_horizontal=0.8,
        relative_scaling=0.55,
        collocations=False,
        margin=4,
        random_state=40
    ).generate_from_frequencies(freq_dict)
    return wc

def plot_wordcloud_en(pos_series, neg_series, title):
    pos_dict = pos_series.to_dict()
    neg_dict = neg_series.to_dict()

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)

    wc_pos = make_wordcloud_en(pos_dict, 'Blues')
    wc_neg = make_wordcloud_en(neg_dict, 'Reds')

    if wc_pos:
        axes[0].imshow(wc_pos, interpolation='bilinear')
    axes[0].set_title('Positive Keywords', fontsize=13, pad=10)
    axes[0].axis('off')

    if wc_neg:
        axes[1].imshow(wc_neg, interpolation='bilinear')
    axes[1].set_title('Negative Keywords', fontsize=13, pad=10)
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
# 전체 리뷰 워드클라우드
plot_wordcloud_en(
    all_results_gb['total']['긍정'],
    all_results_gb['total']['부정'],
    'Global Sunscreen Reviews — Positive & Negative Keywords'
)


In [ ]:
# 제품별 워드클라우드
for product in ['1_product_a', '2_product_b',,,]:
    plot_wordcloud_en(
        all_results_gb[product]['긍정'],
        all_results_gb[product]['부정'],
        f'{product} — Positive & Negative Keywords'
    )
